# MTurk Response Analysis

Process and analyze survey responses from the explanation rating study.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import sys
import textwrap
from collections import Counter, defaultdict
from datetime import datetime
from nltk import word_tokenize
import pandas as pd
from convert_survey_data import load_all_problems,get_line_num_from_selected
DEFAULT_RESPONSES = os.path.join(".", "data", "responses.json")
SURVEY_DATA = os.path.join(".", "data", "survey_data.json")
OUTPUT_DIR = os.path.join(".", "analysis_output")

# Minimum reasoning length (words) to flag low-effort responses
MIN_REASONING_LENGTH = 20


## Helper Functions

In [3]:
def load_responses(path):
    """Load JSONL responses file into a list of dicts."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  ⚠ Skipping malformed line {lineno}: {e}")
    return records


def load_all_explanations(path):
    """Load survey_data.json and build a master set of all (problem_index, explanation_index) pairs."""
    with open(path, "r", encoding="utf-8") as f:
        problems = json.load(f)
    all_expls = []
    for p in problems:
        pid = p["problem_id"]
        for eidx, etext in enumerate(p["explanations"]):
            all_expls.append({
                "problem_index": pid,
                "explanation_index": eidx,
                "explanation_text": etext,
                "selected_line": p.get("selected_line", ""),
                "line_number": p.get("line_number"),
                "problem_key": p.get("problem_key", ""),
            })
    return pd.DataFrame(all_expls)


def flatten_records(records):
    problems = load_all_problems('./problems')
    """Flatten response records into one row per explanation rating."""
    rows = []
    for rec in records:
        base = {
            "worker_id":            rec.get("mturk_worker_id"),
            "completion_code":      rec.get("completion_code"),
            "timestamp_utc":        rec.get("timestamp_utc"),
            "problem_index":        rec.get("problem_index"),
            "problem_id":           rec.get("problem_id"),
            "problem_statement":    rec.get("problem_statement", "")[:120],
            # "line_number":          get_line_num_from_selected(rec.get('selected_line'),problems),
            "selected_line":        rec.get("selected_line"),
            "num_explanations":     rec.get("num_explanations"),
            "java_experience":      rec.get("java_experience_level"),
        }
        demo = rec.get("demographics", {})
        base["demo_experience"] = demo.get("experience")
        base["demo_role"] = demo.get("role")
        base["screener_score"] = demo.get("java_screener_score")
        base["screener_returning"] = demo.get("java_screener_returning", False)

        ratings = rec.get("ratings", [])
        for r in ratings:
            row = dict(base)
            row["explanation_index"] = r.get("explanation_index")
            row["explanation_text"] = r.get("explanation_text", "")[:120]
            row["correctness_rating"] = r.get("correctness_rating")
            row["completeness_rating"] = r.get("completeness_rating")
            row["reasoning"] = r.get("reasoning", "")

            reasoning = row["reasoning"] or ""
            row["reasoning_length"] = len(word_tokenize(reasoning))
            row["flag_short_reasoning"] = len(word_tokenize(reasoning.strip())) < MIN_REASONING_LENGTH
            row["flag_copy_paste"] = _is_likely_copy_paste(
                reasoning, r.get("explanation_text", "")
            )
            rows.append(row)
    return rows


def _is_likely_copy_paste(reasoning, explanation_text):
    """Heuristic: check if reasoning is suspiciously similar to the explanation."""
    if not reasoning or not explanation_text:
        return False
    reason_words = set(word_tokenize(reasoning.lower()))
    expl_words = set(word_tokenize(explanation_text.lower()))
    if not reason_words or not expl_words:
        return False
    overlap = len(reason_words & expl_words) / len(reason_words)
    return overlap > 0.6


## Analysis Functions

In [4]:
def print_header(title):
    width = 70
    print()
    print("═" * width)
    print(f"  {title}")
    print("═" * width)


def analyze_overview(df, records):
    print_header("OVERVIEW")
    n_submissions = len(records)
    print(f"  Total submissions in responses.json:  {n_submissions}")
    print(f"  Total explanation ratings:  {len(df)}")
    print(f"  Unique workers:             {df['worker_id'].nunique()}")
    print(f"  Unique problems rated:      {df['problem_index'].nunique()}")
    if len(df):
        expl_counts = df.groupby(["worker_id", "problem_index"]).size()
        print(f"  Explanations per submission: "
              f"min={expl_counts.min()}, max={expl_counts.max()}, avg={expl_counts.mean():.1f}")
    if "timestamp_utc" in df.columns and df["timestamp_utc"].notna().any():
        times = pd.to_datetime(df["timestamp_utc"], errors="coerce").dropna()
        if len(times):
            print(f"  Date range:                 "
                  f"{times.min():%Y-%m-%d %H:%M} → {times.max():%Y-%m-%d %H:%M} UTC")


def analyze_ratings(df):
    print_header("RATING DISTRIBUTIONS")
    print(f"  Total ratings: {len(df)}")
    if not len(df):
        return
    print("\n  Correctness:")
    for val, count in df["correctness_rating"].value_counts().items():
        print(f"    {val}: {count} ({count/len(df)*100:.1f}%)")
    print("\n  Completeness:")
    for val, count in df["completeness_rating"].value_counts().items():
        print(f"    {val}: {count} ({count/len(df)*100:.1f}%)")
    if df["correctness_rating"].notna().any() and df["completeness_rating"].notna().any():
        print("\n  Correctness × Completeness cross-tab:")
        ct = pd.crosstab(df["correctness_rating"], df["completeness_rating"])
        print(textwrap.indent(ct.to_string(), "    "))


def analyze_quality_flags(df):
    print_header("RESPONSE QUALITY FLAGS")
    total = len(df)
    if not total:
        print("  No ratings to analyze.")
        return
    short = df["flag_short_reasoning"].sum()
    copy = df["flag_copy_paste"].sum()
    print(f"  Short reasoning (<{MIN_REASONING_LENGTH} words): {short}/{total} ({short/total*100:.1f}%)")
    print(f"  Suspected copy-paste reasoning:    {copy}/{total} ({copy/total*100:.1f}%)")
    lengths = df["reasoning_length"]
    print(f"\n  Reasoning length (words):")
    print(f"    Mean:   {lengths.mean():.0f}")
    print(f"    Median: {lengths.median():.0f}")
    print(f"    Min:    {lengths.min():.0f}")
    print(f"    Max:    {lengths.max():.0f}")
    flagged = df[df["flag_short_reasoning"] | df["flag_copy_paste"]]
    if not flagged.empty:
        print(f"\n  Flagged workers:")
        for wid, group in flagged.groupby("worker_id"):
            flags = []
            if group["flag_short_reasoning"].any():
                flags.append(f"short×{group['flag_short_reasoning'].sum()}")
            if group["flag_copy_paste"].any():
                flags.append(f"copy×{group['flag_copy_paste'].sum()}")
            print(f"    {wid}: {', '.join(flags)}")


def analyze_workers(df):
    print_header("PER-WORKER SUMMARY")
    if not len(df):
        print("  No ratings.")
        return pd.DataFrame()
    rows = []
    for wid, group in df.groupby("worker_id"):
        n_submissions = group.groupby("problem_index").ngroups
        row = {
            "worker_id": wid,
            "problems_rated": n_submissions,
            "total_explanations_rated": len(group),
            "pct_correct": (group["correctness_rating"] == "Correct").mean() * 100,
            "pct_complete": (group["completeness_rating"] == "Complete").mean() * 100,
            "avg_reasoning_len": group["reasoning_length"].mean(),
            "flags_short": group["flag_short_reasoning"].sum(),
            "flags_copy": group["flag_copy_paste"].sum(),
            "experience": group["demo_experience"].iloc[0],
            "role": group["demo_role"].iloc[0],
        }
        rows.append(row)
    worker_df = pd.DataFrame(rows).sort_values("total_explanations_rated", ascending=False)
    print(f"  Workers: {len(worker_df)}")
    print(f"  Problems per worker:")
    print(f"    Mean:   {worker_df['problems_rated'].mean():.1f}")
    print(f"    Median: {worker_df['problems_rated'].median():.0f}")
    print(f"    Max:    {worker_df['problems_rated'].max()}")
    display_cols = ["worker_id", "problems_rated", "total_explanations_rated",
                    "pct_correct", "pct_complete", "avg_reasoning_len",
                    "flags_short", "flags_copy"]
    print(f"\n  Worker details:")
    print(textwrap.indent(worker_df[display_cols].to_string(index=False, float_format="%.1f"), "    "))
    return worker_df


def analyze_problems(df):
    print_header("PER-PROBLEM SUMMARY")
    if not len(df):
        print("  No ratings.")
        return pd.DataFrame()
    rows = []
    for pidx, group in df.groupby("problem_index"):
        n_workers = group["worker_id"].nunique()
        n_explanations = group["explanation_index"].nunique()
        row = {
            "problem_index": pidx,
            "num_explanations": n_explanations,
            "num_workers": n_workers,
            "total_ratings": len(group),
            "pct_rated_correct": (group["correctness_rating"] == "Correct").mean() * 100,
            "pct_rated_complete": (group["completeness_rating"] == "Complete").mean() * 100,
            "avg_reasoning_len": group["reasoning_length"].mean(),
            "problem_preview": group["problem_statement"].iloc[0][:80],
        }
        rows.append(row)
    prob_df = pd.DataFrame(rows).sort_values("problem_index")
    print(f"  Problems rated: {len(prob_df)}")
    display_cols = ["problem_index", "num_explanations", "num_workers",
                    "total_ratings", "pct_rated_correct", "pct_rated_complete"]
    print(textwrap.indent(prob_df[display_cols].to_string(index=False, float_format="%.1f"), "    "))
    return prob_df


def analyze_explanations(df):
    """Per-explanation breakdown across all workers."""
    print_header("PER-EXPLANATION SUMMARY")
    if not len(df):
        print("  No ratings.")
        return pd.DataFrame()
    rows = []
    for (pidx, eidx), group in df.groupby(["problem_index", "explanation_index"]):
        raters = sorted(group["worker_id"].unique().tolist())
        n_raters = len(raters)
        row = {
            "problem_index": pidx,
            "explanation_index": eidx,
            "num_raters": n_raters,
            "raters": raters,
            "pct_correct": (group["correctness_rating"] == "Correct").mean() * 100,
            "pct_complete": (group["completeness_rating"] == "Complete").mean() * 100,
            "explanation_preview": group["explanation_text"].iloc[0][:80],
        }
        rows.append(row)
    expl_df = pd.DataFrame(rows).sort_values(["problem_index", "explanation_index"])
    print(f"  Unique explanations rated: {len(expl_df)}")
    display_cols = ["problem_index", "explanation_index", "num_raters",
                    "pct_correct", "pct_complete"]
    print(textwrap.indent(expl_df[display_cols].to_string(index=False, float_format="%.1f"), "    "))
    return expl_df


def compute_inter_rater(df):
    """Compute simple agreement for explanations rated by multiple workers."""
    print_header("INTER-RATER AGREEMENT")
    if not len(df):
        print("  No ratings.")
        return
    multi = df.groupby(["problem_index", "explanation_index"]).filter(
        lambda g: g["worker_id"].nunique() >= 2
    )
    if multi.empty:
        print("  Not enough overlapping ratings (need ≥2 workers per explanation).")
        return
    groups = multi.groupby(["problem_index", "explanation_index"])
    agree_correct = 0
    agree_complete = 0
    total_pairs = 0
    for _, group in groups:
        ratings_c = group["correctness_rating"].dropna().tolist()
        ratings_m = group["completeness_rating"].dropna().tolist()
        n = len(ratings_c)
        for i in range(n):
            for j in range(i + 1, n):
                total_pairs += 1
                if ratings_c[i] == ratings_c[j]:
                    agree_correct += 1
                if i < len(ratings_m) and j < len(ratings_m) and ratings_m[i] == ratings_m[j]:
                    agree_complete += 1
    if total_pairs:
        n_expls = groups.ngroups
        print(f"  Explanations with ≥2 raters: {n_expls}")
        print(f"  Total rater pairs:           {total_pairs}")
        print(f"  Correctness agreement:       "
              f"{agree_correct}/{total_pairs} ({agree_correct/total_pairs*100:.1f}%)")
        print(f"  Completeness agreement:      "
              f"{agree_complete}/{total_pairs} ({agree_complete/total_pairs*100:.1f}%)")


def export_csvs(df, worker_df, prob_df, expl_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    all_path = os.path.join(output_dir, "all_ratings.csv")
    df.to_csv(all_path, index=False)
    worker_path = os.path.join(output_dir, "worker_quality.csv")
    worker_df.to_csv(worker_path, index=False)
    prob_path = os.path.join(output_dir, "problem_summary.csv")
    prob_df.to_csv(prob_path, index=False)
    expl_path = os.path.join(output_dir, "explanation_summary.csv")
    expl_df.to_csv(expl_path, index=False)
    return all_path, worker_path, prob_path, expl_path


## Load Data & Run Analysis

In [5]:
path = DEFAULT_RESPONSES

workers_to_exclude = ["A6A0IS7L6OUAB","AKERE2HC8Y7H4",
                    "AKERE2HC8Y7H3","AKERE2HC8Y7H5",
                    "AKERE2HC8Y7H9",
                    "AKERE2HC8Y7H6",
                    "AKERE2HC8Y7H7",
                        "AM6XJQNLFEV4J",
                        "A3IDTSOSHDX6JK",
                        "ANBCTN6HV7Y1M","A47LXYTWFIB6F",
                        "A7D1G6VBRTJMJ",
                        "A19HN15YVY9E5I","A1A9FWTXG5F5F9","A16KWRV7HCOR92","A1GM18CT92RSX6",
                        "A3N33WTGLSETKN","A1ASIOBX12Z73I"
                    ]

workers_accepted = [ "A3QKH0MDY2P02I","A11ANIKYMXV8G6","A22QWDTMB7MA3D","A3IDRTM133H0VQ",
                    "A2NZLZLLAZYXJX","A2MHDAPN4DVTG4","A2U7UDW9Y1WGWF","A2S4L806KJY42M",
                    "A328Z2N5OQRU1T","A2JEPT85S2FRRN","A1YTFCLA5B1VT9",
                    "A2FK6782M09EBO","A3N9HNU9SOX3CC","A2P8LADS527NHD","A2QNDIZF2O1EYW","A2S4L806KJY42M",
                    "A2NZLZLLAZYXJX","AC433OK4T7R5B","A3OZWZHM77YXS1","A26211PR4VJSYZ",
                    "AQCV15GPERWPW"]

workers_to_review = [""]

print(f"Loading responses from: {path}")
records = load_responses(path)
print(f"  {len(records)} submissions loaded")

flat_rows = flatten_records(records)
df = pd.DataFrame(flat_rows)

# Optional filtering — uncomment one:
# df = df[df.worker_id.isin(workers_to_review)]
df = df[df.worker_id.isin(workers_accepted)]
# df = df[~df.worker_id.isin(workers_to_exclude)]

print(f"  {len(df)} ratings after filtering")


Loading responses from: ./data/responses.json
  91 submissions loaded
  208 ratings after filtering


In [7]:
analyze_overview(df, records)
analyze_ratings(df)
analyze_quality_flags(df)
worker_df = analyze_workers(df)
prob_df = analyze_problems(df)
expl_df = analyze_explanations(df)
compute_inter_rater(df)



══════════════════════════════════════════════════════════════════════
  OVERVIEW
══════════════════════════════════════════════════════════════════════
  Total submissions in responses.json:  91
  Total explanation ratings:  208
  Unique workers:             19
  Unique problems rated:      18
  Explanations per submission: min=8, max=8, avg=8.0
  Date range:                 2026-03-17 05:41 → 2026-03-18 12:19 UTC

══════════════════════════════════════════════════════════════════════
  RATING DISTRIBUTIONS
══════════════════════════════════════════════════════════════════════
  Total ratings: 208

  Correctness:
    Correct: 179 (86.1%)
    Incorrect: 29 (13.9%)

  Completeness:
    Incomplete: 180 (86.5%)
    Complete: 28 (13.5%)

  Correctness × Completeness cross-tab:
    completeness_rating  Complete  Incomplete
    correctness_rating                       
    Correct                    26         153
    Incorrect                   2          27

══════════════════════════════

In [8]:
all_path, worker_path, prob_path, expl_path = export_csvs(
    df, worker_df, prob_df, expl_df, OUTPUT_DIR
)
print_header("EXPORTS")
print(f"  All ratings:           {all_path}")
print(f"  Worker quality:        {worker_path}")
print(f"  Problem summary:       {prob_path}")
print(f"  Explanation summary:   {expl_path}")



══════════════════════════════════════════════════════════════════════
  EXPORTS
══════════════════════════════════════════════════════════════════════
  All ratings:           ./analysis_output/all_ratings.csv
  Worker quality:        ./analysis_output/worker_quality.csv
  Problem summary:       ./analysis_output/problem_summary.csv
  Explanation summary:   ./analysis_output/explanation_summary.csv


## Detailed Rating View

In [10]:
df[['worker_id','timestamp_utc','selected_line','explanation_text',
    'reasoning','correctness_rating','completeness_rating']].head(20)


,worker_id,timestamp_utc,selected_line,explanation_text,reasoning,correctness_rating,completeness_rating
16,A2S4L806KJY42M,2026-03-17T05:41:11.705815+00:00,/*6*/ Scanner scan = new Scanner(System.in);,This creates a scanner object names scan so th...,The explanation is correct because it accurate...,Correct,Incomplete
17,A2S4L806KJY42M,2026-03-17T05:41:11.705815+00:00,/*6*/ Scanner scan = new Scanner(System.in);,The usage of scanner is to take read what the ...,The explanation is correct because it generall...,Correct,Incomplete
18,A2S4L806KJY42M,2026-03-17T05:41:11.705815+00:00,/*6*/ Scanner scan = new Scanner(System.in);,imports scanner to take a users input,The explanation is incorrect because it refers...,Incorrect,Incomplete
19,A2S4L806KJY42M,2026-03-17T05:41:11.705815+00:00,/*6*/ Scanner scan = new Scanner(System.in);,This line initializes a Scanner object that in...,The explanation is correct because it accurate...,Correct,Incomplete
20,A2S4L806KJY42M,2026-03-17T05:41:11.705815+00:00,/*6*/ Scanner scan = new Scanner(System.in);,Defines a scanner object named scan.,The explanation is correct because it accurate...,Correct,Incomplete
21,A2S4L806KJY42M,2026-03-17T05:41:11.705815+00:00,/*6*/ Scanner scan = new Scanner(System.in);,This line creates an object names scan of the ...,The explanation is correct because it accurate...,Correct,Incomplete
22,A2S4L806KJY42M,2026-03-17T05:41:11.705815+00:00,/*6*/ Scanner scan = new Scanner(System.in);,initiates an object of the scanner function to...,Q3:\nThe explanation is correct because it con...,Correct,Incomplete
23,A2S4L806KJY42M,2026-03-17T05:41:11.705815+00:00,/*6*/ Scanner scan = new Scanner(System.in);,scans keyboard for inputs,Q3:\nThe explanation is correct because it con...,Correct,Incomplete
136,A3QKH0MDY2P02I,2026-03-17T16:50:32.593392+00:00,"/*6*/ point.translate(11, 6);",uses point object to call function translate o...,The explanation is incorrect because it wrongl...,Incorrect,Incomplete
137,A3QKH0MDY2P02I,2026-03-17T16:50:32.593392+00:00,"/*6*/ point.translate(11, 6);",The method translate is used to manipulate int...,The explanation is incorrect because it vaguel...,Incorrect,Incomplete


## Explanation Coverage Analysis

Which explanations are still unrated? How many raters does each have?


In [11]:
# Load the master list of ALL explanations from survey_data.json
all_expls_df = load_all_explanations(SURVEY_DATA)
print(f"Total explanations in survey_data.json: {len(all_expls_df)}")

# Build rated explanations summary with rater lists
if len(df):
    rated_summary = (
        df.groupby(["problem_index", "explanation_index"]).agg(
            num_raters=("worker_id", "nunique"),
            raters=("worker_id", lambda x: sorted(x.unique().tolist())),
        ).reset_index()
    )

else:
    rated_summary = pd.DataFrame(columns=["problem_index", "explanation_index", "num_raters", "raters"])

# Merge to find coverage
coverage = all_expls_df.merge(
    rated_summary,
    on=["problem_index", "explanation_index"],
    how="left",
)
coverage["num_raters"] = coverage["num_raters"].fillna(0).astype(int)
coverage["raters"] = coverage["raters"].apply(lambda x: x if isinstance(x, list) else [])

print(f"Explanations with ratings: {(coverage['num_raters'] > 0).sum()}")
print(f"Explanations unrated:      {(coverage['num_raters'] == 0).sum()}")


Total explanations in survey_data.json: 216
Explanations with ratings: 144
Explanations unrated:      72


### Unrated Explanations

In [12]:
unrated = coverage[coverage["num_raters"] == 0][
    ["problem_index", "explanation_index", "problem_key","line_number","selected_line", "explanation_text"]
]
print(f"Total unrated: {len(unrated)}")
unrated


Total unrated: 72


,problem_index,explanation_index,problem_key,line_number,selected_line,explanation_text
0,0,0,JAdjacentDuplicates.java,5,"/*5*/ System.out.println(""Enter the numbers...",This line asks the user to enter numbers that ...
1,0,1,JAdjacentDuplicates.java,5,"/*5*/ System.out.println(""Enter the numbers...","this will print out a new line that says ""ente..."
2,0,2,JAdjacentDuplicates.java,5,"/*5*/ System.out.println(""Enter the numbers...","prints ""Enter the numbers separated by spaces ..."
3,0,3,JAdjacentDuplicates.java,5,"/*5*/ System.out.println(""Enter the numbers...",The line here prompts the user prints out to c...
4,0,4,JAdjacentDuplicates.java,5,"/*5*/ System.out.println(""Enter the numbers...","Prints out the statement ""Enter the numbers se..."
...,...,...,...,...,...,...
123,15,3,JSearchArrayValues.java,8,/*8*/ if (val1 == val2) {,The line here is the innermost loop of the nes...
124,15,4,JSearchArrayValues.java,8,/*8*/ if (val1 == val2) {,An if statement which executes the code below ...
125,15,5,JSearchArrayValues.java,8,/*8*/ if (val1 == val2) {,This line creates an if statement that checks ...
126,15,6,JSearchArrayValues.java,8,/*8*/ if (val1 == val2) {,if val1 is equal to val2 it is true


In [13]:
unrated.rename(columns={'explanation_text':'explanation'}).to_csv('./data/unrated.csv',index=False)

### Explanations by Number of Raters

In [13]:
# Distribution of rater counts
rater_dist = coverage["num_raters"].value_counts().sort_index()
print("Rater count distribution:")
for count, n in rater_dist.items():
    print(f"  {count} rater(s): {n} explanation(s)")

print()

# Show explanations grouped by number of raters
for n_raters in sorted(coverage["num_raters"].unique()):
    subset = coverage[coverage["num_raters"] == n_raters]
    label = "unrated" if n_raters == 0 else f"{n_raters} rater(s)"
    print(f"\n{'─'*60}")
    print(f"  {label}: {len(subset)} explanation(s)")
    print(f"{'─'*60}")
    display_cols = ["problem_index", "explanation_index", "problem_key",
                    "selected_line", "num_raters", "raters"]
    avail_cols = [c for c in display_cols if c in subset.columns]
    print(subset[avail_cols].to_string(index=False))


Rater count distribution:
  0 rater(s): 72 explanation(s)
  1 rater(s): 96 explanation(s)
  2 rater(s): 32 explanation(s)
  3 rater(s): 16 explanation(s)


────────────────────────────────────────────────────────────
  unrated: 72 explanation(s)
────────────────────────────────────────────────────────────
 problem_index  explanation_index              problem_key                                                                                          selected_line  num_raters raters
             0                  0 JAdjacentDuplicates.java /*5*/    System.out.println("Enter the numbers separated by spaces or new line (enter -1 to quit): ");           0     []
             0                  1 JAdjacentDuplicates.java /*5*/    System.out.println("Enter the numbers separated by spaces or new line (enter -1 to quit): ");           0     []
             0                  2 JAdjacentDuplicates.java /*5*/    System.out.println("Enter the numbers separated by spaces or new line (enter -1 to

### Raters per Explanation (Aggregated)

In [14]:
# Full table: every explanation with its rater list
raters_view = coverage[
    ["problem_index", "explanation_index", "problem_key",
     "selected_line", "explanation_text", "num_raters", "raters"]
].sort_values(["problem_index", "explanation_index"])

# Convert raters list to comma-separated string for display
raters_view = raters_view.copy()
raters_view["raters"] = raters_view["raters"].apply(lambda x: ", ".join(x) if x else "—")

raters_view


,problem_index,explanation_index,problem_key,selected_line,explanation_text,num_raters,raters
0,0,0,JAdjacentDuplicates.java,"/*5*/ System.out.println(""Enter the numbers...",This line asks the user to enter numbers that ...,0,—
1,0,1,JAdjacentDuplicates.java,"/*5*/ System.out.println(""Enter the numbers...","this will print out a new line that says ""ente...",0,—
2,0,2,JAdjacentDuplicates.java,"/*5*/ System.out.println(""Enter the numbers...","prints ""Enter the numbers separated by spaces ...",0,—
3,0,3,JAdjacentDuplicates.java,"/*5*/ System.out.println(""Enter the numbers...",The line here prompts the user prints out to c...,0,—
4,0,4,JAdjacentDuplicates.java,"/*5*/ System.out.println(""Enter the numbers...","Prints out the statement ""Enter the numbers se...",0,—
...,...,...,...,...,...,...,...
211,26,3,PointTester.java,/*23*/ public int getX() {,This line serves to initiate the public getter...,1,A1YTFCLA5B1VT9
212,26,4,PointTester.java,/*23*/ public int getX() {,defines a get method which returns x.,1,A1YTFCLA5B1VT9
213,26,5,PointTester.java,/*23*/ public int getX() {,This line creates a method called getX. This h...,1,A1YTFCLA5B1VT9
214,26,6,PointTester.java,/*23*/ public int getX() {,declares a method getX which returns an integer,1,A1YTFCLA5B1VT9
